# TekaRx Random Forest Baseline

This notebook trains a **Random Forest** classifier as a baseline model for the TekaRx drug safety prediction task. It uses the same leakage-controlled FAERS experiment setup with **2019Q1–2023Q4 for training, 2024Q1 for validation, and 2024Q2 held out for final testing**.

> TekaRx outputs are research decision-support signals, not diagnoses or clinical advice. FAERS reports do not establish causality.

## Overview

This notebook:
1. Loads the pre-processed graph features from the TekaRx pipeline
2. Extracts patient-level features (excluding drug neighbor aggregation)
3. Trains a Random Forest classifier with proper train/val/test splits
4. Evaluates performance using AUC-ROC metrics
5. Saves the trained model and feature importances

## 1. Setup and Dependencies

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib seaborn

In [ ]:
import json
import pickle
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    classification_report,
    confusion_matrix
)

print("Dependencies loaded successfully")

## 2. Configuration

In [ ]:
# Data directory configuration
DATA_DIR = Path("data")  # or set via environment: os.getenv("TEKARX_DATA_DIR", "data")
GRAPH_PATH = DATA_DIR / "processed" / "tekarx_graph.pt"
OUTPUT_DIR = DATA_DIR / "processed" / "randomforest"

# Model hyperparameters
RF_CONFIG = {
    "n_estimators": 100,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "bootstrap": True,
    "class_weight": "balanced",  # Handle class imbalance
    "random_state": 42,
    "n_jobs": -1,  # Use all available cores
}

# Feature track options: "prospective", "prospective-no-dosage", "completed_report"
FEATURE_TRACK = "prospective"

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

## 3. Load Preprocessed Data

In [ ]:
def load_tekarx_graph(graph_path: Path):
    """Load the TekaRx graph and extract patient features and labels."""
    try:
        import torch
    except ImportError:
        raise ImportError("PyTorch is required to load .pt files. Install with: pip install torch")
    
    if not graph_path.exists():
        raise FileNotFoundError(f"Graph file not found: {graph_path}")
    
    print(f"Loading graph from {graph_path}...")
    graph_data = torch.load(graph_path, weights_only=False)
    
    return graph_data


def extract_patient_features(graph_data, feature_track="prospective"):
    """Extract patient-level features and labels from the graph."""
    import torch
    
    # Patient node features
    patient_x = graph_data["patient_x"]  # Shape: [num_patients, num_features]
    
    # Get feature names from manifest if available
    feature_names = tuple(graph_data.get("feature_names", []))
    
    # Labels and split information
    labels = graph_data["labels"]  # Binary labels for patients
    split_id = graph_data["split_id"]  # 0=train, 1=val, 2=test
    
    # Convert to numpy if tensors
    if isinstance(patient_x, torch.Tensor):
        patient_x = patient_x.numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.numpy()
    if isinstance(split_id, torch.Tensor):
        split_id = split_id.numpy()
    
    print(f"Patient features shape: {patient_x.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"Split ID shape: {split_id.shape}")
    print(f"\nLabel distribution:")
    print(f"  Negative (0): {(labels == 0).sum()}")
    print(f"  Positive (1): {(labels == 1).sum()}")
    print(f"\nSplit distribution:")
    print(f"  Train: {(split_id == 0).sum()}")
    print(f"  Validation: {(split_id == 1).sum()}")
    print(f"  Test: {(split_id == 2).sum()}")
    
    return patient_x, feature_names, labels, split_id


# Load the graph
if GRAPH_PATH.exists():
    graph_data = load_tekarx_graph(GRAPH_PATH)
    X, feature_names, y, split_id = extract_patient_features(graph_data, FEATURE_TRACK)
else:
    print(f"Graph not found at {GRAPH_PATH}")
    print("Please ensure you have run the graph building pipeline first.")
    # For demonstration, create synthetic data
    np.random.seed(42)
    n_samples = 10000
    n_features = 100
    X = np.random.randn(n_samples, n_features)
    feature_names = tuple([f"feature_{i}" for i in range(n_features)])
    y = np.random.randint(0, 2, n_samples)
    split_id = np.zeros(n_samples, dtype=int)
    split_id[int(n_samples*0.7):int(n_samples*0.85)] = 1
    split_id[int(n_samples*0.85):] = 2
    print("Using synthetic data for demonstration.")

## 4. Prepare Train/Validation/Test Splits

In [ ]:
# Split data according to TekaRx protocol
train_mask = split_id == 0
val_mask = split_id == 1
test_mask = split_id == 2

X_train = X[train_mask]
y_train = y[train_mask]
X_val = X[val_mask]
y_val = y[val_mask]
X_test = X[test_mask]
y_test = y[test_mask]

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

print(f"\nClass balance in training set:")
print(f"  Negative: {(y_train == 0).sum()} ({100*(y_train == 0).mean():.1f}%)")
print(f"  Positive: {(y_train == 1).sum()} ({100*(y_train == 1).mean():.1f}%)")

## 5. Train Random Forest Model

In [ ]:
print("Initializing Random Forest classifier...")
rf_model = RandomForestClassifier(**RF_CONFIG)

print(f"Training with {RF_CONFIG['n_estimators']} trees...")
print(f"Class weight: {RF_CONFIG['class_weight']}")
print(f"Max features: {RF_CONFIG['max_features']}")

rf_model.fit(X_train, y_train)
print("\nTraining complete!")

## 6. Evaluate Model Performance

In [ ]:
def evaluate_model(model, X, y, dataset_name="Dataset"):
    """Comprehensive model evaluation."""
    # Predictions
    y_pred_proba = model.predict_proba(X)[:, 1]
    y_pred = model.predict(X)
    
    # Metrics
    auc_roc = roc_auc_score(y, y_pred_proba)
    auc_pr = average_precision_score(y, y_pred_proba)
    
    print(f"\n{'='*60}")
    print(f"{dataset_name} Performance")
    print(f"{'='*60}")
    print(f"AUC-ROC: {auc_roc:.4f}")
    print(f"AUC-PR:  {auc_pr:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y, y_pred, digits=4))
    
    return {
        "auc_roc": auc_roc,
        "auc_pr": auc_pr,
        "predictions_proba": y_pred_proba,
        "predictions": y_pred
    }


# Evaluate on all splits
train_results = evaluate_model(rf_model, X_train, y_train, "Training")
val_results = evaluate_model(rf_model, X_val, y_val, "Validation")
test_results = evaluate_model(rf_model, X_test, y_test, "Test (Hold-out)")

print(f"\n{'='*60}")
print("Summary")
print(f"{'='*60}")
print(f"Train AUC-ROC:   {train_results['auc_roc']:.4f}")
print(f"Val AUC-ROC:     {val_results['auc_roc']:.4f}")
print(f"Test AUC-ROC:    {test_results['auc_roc']:.4f}")

## 7. Visualization

In [ ]:
plt.figure(figsize=(12, 10))

# ROC Curves
plt.subplot(2, 2, 1)
for results, name in [(train_results, "Train"), (val_results, "Val"), (test_results, "Test")]:
    fpr, tpr, _ = roc_curve(y[[train_mask, val_mask, test_mask][["Train", "Val", "Test"].index(name)]], 
                            results["predictions_proba"])
    plt.plot(fpr, tpr, label=f"{name} (AUC={results['auc_roc']:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

# Precision-Recall Curves
plt.subplot(2, 2, 2)
for results, name in [(train_results, "Train"), (val_results, "Val"), (test_results, "Test")]:
    precision, recall, _ = precision_recall_curve(y[[train_mask, val_mask, test_mask][["Train", "Val", "Test"].index(name)]], 
                                                   results["predictions_proba"])
    plt.plot(recall, precision, label=f"{name} (AP={results['auc_pr']:.3f})")
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves')
plt.legend(loc='lower left')
plt.grid(True, alpha=0.3)

# Prediction Distribution
plt.subplot(2, 2, 3)
sns.histplot(data=pd.DataFrame({
    'Prediction': test_results["predictions_proba"],
    'True Label': y_test.astype(str)
}), x='Prediction', hue='True Label', bins=50, alpha=0.5)
plt.xlabel('Predicted Probability')
plt.ylabel('Count')
plt.title('Test Set Prediction Distribution')
plt.grid(True, alpha=0.3)

# Confusion Matrix
plt.subplot(2, 2, 4)
cm = confusion_matrix(y_test, test_results["predictions"])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Test Set Confusion Matrix')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "evaluation_plots.png", dpi=150, bbox_inches='tight')
print(f"Plots saved to {OUTPUT_DIR / 'evaluation_plots.png'}")
plt.show()

In [ ]:
# Feature Importance Analysis
feature_importance = pd.DataFrame({
    'feature': feature_names if len(feature_names) == X.shape[1] else [f'feature_{i}' for i in range(X.shape[1])],
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features:")
print(feature_importance.head(20).to_string(index=False))

# Plot top features
plt.figure(figsize=(10, 8))
top_n = min(20, len(feature_importance))
sns.barplot(data=feature_importance.head(top_n), x='importance', y='feature', palette='viridis')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title(f'Top {top_n} Feature Importances')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=150, bbox_inches='tight')
print(f"\nFeature importance plot saved to {OUTPUT_DIR / 'feature_importance.png'}")
plt.show()

# Save feature importance to CSV
feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
print(f"Feature importance saved to {OUTPUT_DIR / 'feature_importance.csv'}")

## 8. Save Model and Artifacts

In [ ]:
# Generate timestamp for model versioning
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save the trained model
model_path = OUTPUT_DIR / f"randomforest_{FEATURE_TRACK}_{timestamp}.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(rf_model, f)
print(f"Model saved to {model_path}")

# Save metadata/manifest
manifest = {
    "model_type": "RandomForestClassifier",
    "feature_track": FEATURE_TRACK,
    "timestamp": timestamp,
    "hyperparameters": RF_CONFIG,
    "metrics": {
        "train_auc_roc": float(train_results['auc_roc']),
        "train_auc_pr": float(train_results['auc_pr']),
        "val_auc_roc": float(val_results['auc_roc']),
        "val_auc_pr": float(val_results['auc_pr']),
        "test_auc_roc": float(test_results['auc_roc']),
        "test_auc_pr": float(test_results['auc_pr']),
    },
    "data_info": {
        "n_features": int(X.shape[1]),
        "n_train": int(X_train.shape[0]),
        "n_val": int(X_val.shape[0]),
        "n_test": int(X_test.shape[0]),
        "class_balance_train": float(y_train.mean()),
    },
    "graph_path": str(GRAPH_PATH),
}

manifest_path = OUTPUT_DIR / f"randomforest_{FEATURE_TRACK}_{timestamp}_manifest.json"
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f"Manifest saved to {manifest_path}")

# Save predictions for later analysis
predictions_df = pd.DataFrame({
    'split': ['train']*len(y_train) + ['val']*len(y_val) + ['test']*len(y_test),
    'true_label': np.concatenate([y_train, y_val, y_test]),
    'predicted_proba': np.concatenate([
        train_results['predictions_proba'],
        val_results['predictions_proba'],
        test_results['predictions_proba']
    ]),
    'predicted_class': np.concatenate([
        train_results['predictions'],
        val_results['predictions'],
        test_results['predictions']
    ])
})
predictions_path = OUTPUT_DIR / f"predictions_{FEATURE_TRACK}_{timestamp}.csv"
predictions_df.to_csv(predictions_path, index=False)
print(f"Predictions saved to {predictions_path}")

print("\n" + "="*60)
print("Training Complete!")
print("="*60)
print(f"\nFinal Test AUC-ROC: {test_results['auc_roc']:.4f}")
print(f"Final Test AUC-PR:  {test_results['auc_pr']:.4f}")
print(f"\nAll artifacts saved to: {OUTPUT_DIR}")

## Notes

- This Random Forest baseline provides a non-neural comparison point for the GNN models
- The `class_weight='balanced'` parameter helps handle class imbalance in adverse event reporting
- Feature importances can help identify which patient characteristics are most predictive
- The test set remains completely held-out until final evaluation (no hyperparameter tuning on test)

## Next Steps

1. Compare Random Forest performance against GNN baselines
2. Analyze feature importances for clinical interpretability
3. Consider ensemble methods combining RF and GNN predictions
4. Perform error analysis on misclassified cases